# 📙 Módulo 05 - Notebook 02: Conciliación automática Libro-Extracto

## 🔍 Conciliación avanzada con tolerancia y múltiples criterios

**Libro:** Saliendo de lo Pandito  
**Módulo:** 05 - Reshaping y Conciliaciones  
**Duración estimada:** 65 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Aplicar** conciliación con tolerancia de diferencias  
✅ **Implementar** matching por múltiples criterios (fecha + monto)  
✅ **Usar** algoritmos de coincidencia fuzzy  
✅ **Generar** reportes ejecutivos automatizados  
✅ **Clasificar** partidas conciliadas, pendientes y diferencias

---

## 📋 Pre-requisitos

* ✅ Notebook 05_01 completado (Merge y Join básico)
* ✅ Conocimiento de pd.merge() y tipos de join
* ✅ Familiaridad con conciliaciones manuales

---

## 📚 Contenido

1. Conciliación con Tolerancia de Diferencias
2. Matching por Múltiples Criterios
3. Algoritmos de Coincidencia Fuzzy
4. Clasificación de Estados
5. Reporte Ejecutivo Automatizado
6. Caso Integrador: Conciliación Bancaria Completa

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Simular dos fuentes para conciliación: libro contable y extracto bancario
    # El libro tiene TODOS los movimientos
    libro = df[['fecha', 'sucursal_id', 'sucursal_nombre', 'ventas']].copy()
    libro['id_mov'] = range(1, len(libro) + 1)
    
    # El extracto tiene 95% de los movimientos (simula partidas faltantes)
    extracto = libro.sample(frac=0.95, random_state=42)[['fecha', 'sucursal_id', 'ventas']].copy()
    extracto['id_banco'] = range(1, len(extracto) + 1)
    
    # Introducir pequeñas diferencias de redondeo (simula errores de carga)
    np.random.seed(42)
    mask = np.random.rand(len(extracto)) < 0.10  # 10% con diferencias
    extracto.loc[mask, 'ventas'] = extracto.loc[mask, 'ventas'] + np.random.uniform(-0.5, 0.5, mask.sum())
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros libro: {len(libro):,}")
    print(f"   🏬 Registros extracto: {len(extracto):,}")
    print(f"   ⚠️  Diferencia: {len(libro) - len(extracto)} partidas sin match")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    print(f"\n📋 Estructura de datos:")
    print(f"   • libro: DataFrame con columnas [fecha, sucursal_id, sucursal_nombre, ventas, id_mov]")
    print(f"   • extracto: DataFrame con columnas [fecha, sucursal_id, ventas, id_banco]")
    
    print(f"\n🎯 Este notebook usará datos REALES para conciliación")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    libro = None
    extracto = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Conciliación avanzada: Más allá del match exacto

### 🔍 El problema del mundo real

En el Notebook 05_01 vimos conciliación **exacta**: un movimiento en el libro coincide perfectamente con uno en el extracto.

**Pero en el mundo real:**

❌ **Redondeos:** $1,234.567 vs $1,234.57  
❌ **Timing:** Mismo día pero horas diferentes  
❌ **Referencias:** "FAC-001" vs "Factura 001"  
❌ **Errores de carga:** $1,500.00 vs $1,500.50

---

### ⚖️ Solución: Tolerancia y múltiples criterios

#### 1️⃣ **Tolerancia de Diferencias**

```python
# En vez de:
monto_libro == monto_banco

# Usamos:
abs(monto_libro - monto_banco) <= TOLERANCIA  # Ej: $1.00
```

**Caso de uso:** Diferencias de centavos por redondeos

---

#### 2️⃣ **Matching Multi-Criterio**

```python
# Coincidir por:
- Fecha (+/- 1 día)
- Monto (con tolerancia)
- Referencia (opcional)
```

**Caso de uso:** Movimientos con fechas levemente diferentes

---

#### 3️⃣ **Clasificación de Estados**

| Estado | Descripción | Acción |
|--------|-------------|----------|
| ✅ **Conciliado** | Match exacto o con tolerancia | Ninguna |
| ⚠️ **Pendiente** | Solo en libro | Investigar |
| ⚠️ **Sobrante** | Solo en extracto | Verificar |
| ❌ **Diferencia** | Montos diferentes > tolerancia | Ajustar |

---

### 📋 Proceso de Conciliación automática

```
1. Merge OUTER (libro + extracto)
2. Calcular diferencias absolutas
3. Clasificar según tolerancia
4. Generar reporte ejecutivo
5. Exportar partidas pendientes
```

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("🔍 CONCILIACIÓN AVANZADA: Fundamentos")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

print("\n📖 Técnicas de este notebook:")
print("  1. Tolerancia de diferencias (absoluta y porcentual)")
print("  2. Matching multi-criterio (fecha + monto + ref)")
print("  3. Clasificación automática de estados")
print("  4. Reporte ejecutivo con métricas")

print("\n🎯 Métodos clave:")
print("  - pd.merge(..., how='outer', indicator=True)")
print("  - abs(diff) <= tolerancia")
print("  - .apply() para lógica de clasificación")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## ⚖️ Conciliación con tolerancia: Datos reales de Los Andes Market

### 🎯 La tolerancia en el contexto real

En la celda de carga de datos, simulamos dos fuentes:
* **Libro:** todos los registros de `ventas_mensuales_mendoza_h3`
* **Extracto:** 95% de los registros, con 10% teniendo diferencias de redondeo

**¿Por qué usar tolerancia?**

Los datos reales de Los Andes Market tienen valores decimales en `ventas`. Cuando el banco registra el depósito, puede haber diferencias de centavos por:
* Redondeo bancario
* Comisiones no documentadas
* Errores de tipeo en la carga

---
### 📐 Tipos de tolerancia

```python
# Tolerancia absoluta: $1 de diferencia
TOLERANCIA_ABS = 1.0
abs(diff) <= TOLERANCIA_ABS

# Tolerancia porcentual: 1% del monto
TOLERANCIA_PCT = 0.01
abs(diff) / monto_libro <= TOLERANCIA_PCT

# Tolerancia combinada: la que sea menor
abs(diff) <= min(TOLERANCIA_ABS, monto_libro * TOLERANCIA_PCT)
```

---

### 💡 Aplicación a Los Andes Market

Con montos de ventas entre $80,000 y $200,000:
* Tolerancia absoluta de $1: captura redondeos pero no errores de carga
* Tolerancia porcentual de 1%: permite hasta $2,000 de diferencia (demasiado)
* **Recomendado:** tolerancia absoluta de $1 + clasificar por severidad

In [0]:
import pandas as pd
import numpy as np

print("⚖️ CONCILIACIÓN CON TOLERANCIA — DATOS REALES")
print("="*70)

if USAR_DATOS_REALES and df is not None:
    # Usar libro y extracto ya creados en la celda de carga
    print(f"\n📚 Libro: {len(libro):,} registros")
    print(f"🏬 Extracto: {len(extracto):,} registros")

    print("\n" + "="*70)
    print("\n1️⃣  CONCILIACIÓN SIN TOLERANCIA (estricto)")
    print("-"*70)

    conc_estricto = pd.merge(
        libro[['id_mov', 'sucursal_id', 'fecha', 'ventas']],
        extracto[['id_banco', 'sucursal_id', 'fecha', 'ventas']],
        on=['sucursal_id', 'fecha'],
        how='outer',
        indicator=True,
        suffixes=('_libro', '_banco')
    )
    conc_estricto['diferencia'] = abs(
        conc_estricto['ventas_libro'].fillna(0) - conc_estricto['ventas_banco'].fillna(0)
    )
    conc_estricto['estado'] = conc_estricto.apply(
        lambda r: '✅ Conciliado' if r['_merge'] == 'both' and r['diferencia'] == 0
        else '❌ Diferencia' if r['_merge'] == 'both'
        else '⚠️ Solo Libro' if r['_merge'] == 'left_only'
        else '⚠️ Solo Extracto', axis=1
    )
    print(conc_estricto['estado'].value_counts())
    estictos_ok = (conc_estricto['estado'] == '✅ Conciliado').sum()
    print(f"\n   Tasa de conciliación estricta: {estictos_ok/len(conc_estricto)*100:.1f}%")

    print("\n" + "="*70)
    print("\n2️⃣  CONCILIACIÓN CON TOLERANCIA ($1)")
    print("-"*70)

    TOLERANCIA = 1.0
    conc_tol = conc_estricto.copy()
    conc_tol['estado'] = conc_tol.apply(
        lambda r: '✅ Conciliado' if r['_merge'] == 'both' and r['diferencia'] <= TOLERANCIA
        else '❌ Diferencia' if r['_merge'] == 'both'
        else '⚠️ Solo Libro' if r['_merge'] == 'left_only'
        else '⚠️ Solo Extracto', axis=1
    )
    print(conc_tol['estado'].value_counts())
    tol_ok = (conc_tol['estado'] == '✅ Conciliado').sum()
    print(f"\n   Tasa con tolerancia ${TOLERANCIA}: {tol_ok/len(conc_tol)*100:.1f}%")
    print(f"   Mejora: +{(tol_ok - estictos_ok)} registros conciliados")

    print("\n" + "="*70)
    print("\n3️⃣  CONCILIACIÓN CON TOLERANCIA PORCENTUAL (0.01%)")
    print("-"*70)

    TOLERANCIA_PCT = 0.0001  # 0.01%
    conc_pct = conc_estricto.copy()
    conc_pct['tol_pct'] = conc_pct['ventas_libro'].fillna(0) * TOLERANCIA_PCT
    conc_pct['estado'] = conc_pct.apply(
        lambda r: '✅ Conciliado' if r['_merge'] == 'both' and r['diferencia'] <= max(r['tol_pct'], 0.01)
        else '❌ Diferencia' if r['_merge'] == 'both'
        else '⚠️ Solo Libro' if r['_merge'] == 'left_only'
        else '⚠️ Solo Extracto', axis=1
    )
    print(conc_pct['estado'].value_counts())

    print("\n" + "="*70)
    print("\n4️⃣  CLASIFICACIÓN POR SEVERIDAD")
    print("-"*70)

    conc_tol['severidad'] = 'Normal'
    conc_tol.loc[(conc_tol['diferencia'] > 1) & (conc_tol['diferencia'] <= 10), 'severidad'] = '🟡 Baja'
    conc_tol.loc[(conc_tol['diferencia'] > 10) & (conc_tol['diferencia'] <= 100), 'severidad'] = '🟠 Media'
    conc_tol.loc[conc_tol['diferencia'] > 100, 'severidad'] = '🔴 Alta'

    diferencias = conc_tol[conc_tol['estado'] == '❌ Diferencia']
    if len(diferencias) > 0:
        print(f"\n   {len(diferencias)} diferencias clasificadas por severidad:")
        print(diferencias[['sucursal_id', 'fecha', 'ventas_libro', 'ventas_banco',
                           'diferencia', 'severidad']].head(10).round(2))
        print(f"\n   Distribución por severidad:")
        print(diferencias['severidad'].value_counts())
else:
    print("\n⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

## 🔗 Matching multi-criterio y reporte ejecutivo con datos reales

### 🎯 ¿Por qué múltiples criterios?

En la conciliación con datos de Los Andes Market, usar solo `sucursal_id` como clave genera duplicados: una sucursal tiene ventas todos los meses. Necesitamos una **clave compuesta** que identifique cada movimiento de forma única.

**Criterios de matching:**
1. `sucursal_id` + `fecha` → identificación exacta del movimiento
2. `ventas` con tolerancia → verificación del monto
3. Diferencia de fecha ≤ 1 día → capturar timing differences

---

### 📊 Reporte ejecutivo automatizado

El reporte ejecutivo debe responder 4 preguntas:

| Pregunta | KPI |
|----------|-----|
| ¿Qué porcentaje se concilió? | Tasa de conciliación |
| ¿Cuántas partidas pendientes hay? | Pendientes en banco y libro |
| ¿Cuál es el monto en disputa? | Suma de diferencias |
| ¿Hay patrones por sucursal? | Diferencias agrupadas por sucursal |

---

### 🛠️ Pipeline de reporte

```python
# 1. Calcular KPIs
kpi_conciliacion = (estado == '✅ Conciliado').sum() / total * 100
kpi_pendientes = (estado == '⚠️ Solo Libro').sum()
kpi_monto_disputa = diferencias['diferencia'].sum()

# 2. Análisis por sucursal
por_sucursal = conciliacion.groupby('sucursal_id')['estado'].value_counts()

# 3. Exportar partidas pendientes para investigación
pendientes.to_csv('pendientes_investigacion.csv')
```

In [0]:
import pandas as pd

print("🔗 MATCHING MULTI-CRITERIO Y REPORTE EJECUTIVO")
print("="*70)

if USAR_DATOS_REALES and df is not None:
    print("\n1️⃣  MATCHING CON CLAVE COMPUESTA: sucursal_id + fecha")
    print("-"*70)

    # Verificar que la clave compuesta es única
    clave_unica = libro.groupby(['sucursal_id', 'fecha']).ngroups
    total_registros = len(libro)
    print(f"\n   Registros: {total_registros:,}")
    print(f"   Claves únicas (sucursal_id + fecha): {clave_unica:,}")
    if clave_unica == total_registros:
        print("   ✅ La clave compuesta es única — no hay duplicados")
    else:
        print(f"   ⚠️  Hay {total_registros - clave_unica} duplicados")

    print("\n" + "="*70)
    print("\n2️⃣  CONCILIACIÓN COMPLETA CON CLAVE COMPUESTA")
    print("-"*70)

    TOLERANCIA = 1.0

    conciliacion_final = pd.merge(
        libro[['id_mov', 'sucursal_id', 'sucursal_nombre', 'fecha', 'año', 'mes', 'ventas']],
        extracto[['id_banco', 'sucursal_id', 'fecha', 'ventas']],
        on=['sucursal_id', 'fecha'],
        how='outer',
        indicator=True,
        suffixes=('_libro', '_banco')
    )

    conciliacion_final['diferencia'] = abs(
        conciliacion_final['ventas_libro'].fillna(0) - conciliacion_final['ventas_banco'].fillna(0)
    )

    def clasificar_final(row):
        if row['_merge'] == 'both':
            if row['diferencia'] <= TOLERANCIA:
                return '✅ Conciliado'
            else:
                return '❌ Diferencia'
        elif row['_merge'] == 'left_only':
            return '⚠️ Pendiente en Banco'
        else:
            return '⚠️ Pendiente en Libro'

    conciliacion_final['estado'] = conciliacion_final.apply(clasificar_final, axis=1)

    print("\n   Muestra de la conciliación:")
    print(conciliacion_final[['sucursal_id', 'sucursal_nombre', 'fecha',
                             'ventas_libro', 'ventas_banco', 'diferencia', 'estado']].head(15).round(2))

    print("\n" + "="*70)
    print("\n3️⃣  REPORTE EJECUTIVO AUTOMATIZADO")
    print("-"*70)

    total = len(conciliacion_final)
    conciliados = (conciliacion_final['estado'] == '✅ Conciliado').sum()
    pend_banco = (conciliacion_final['estado'] == '⚠️ Pendiente en Banco').sum()
    pend_libro = (conciliacion_final['estado'] == '⚠️ Pendiente en Libro').sum()
    diferencias = conciliacion_final[conciliacion_final['estado'] == '❌ Diferencia']

    print(f"\n   📊 KPIs DE CONCILIACIÓN")
    print(f"   {'─'*40}")
    print(f"   Total de movimientos:     {total:>8,}")
    print(f"   ✅ Conciliados:           {conciliados:>8,} ({conciliados/total*100:.1f}%)")
    print(f"   ⚠️  Pendientes en Banco:   {pend_banco:>8,}")
    print(f"   ⚠️  Pendientes en Libro:  {pend_libro:>8,}")
    print(f"   ❌ Diferencias de monto:  {len(diferencias):>8,}")

    if len(diferencias) > 0:
        monto_disputa = diferencias['diferencia'].sum()
        print(f"   💰 Monto en disputa:      ${monto_disputa:>10,.2f}")

    print(f"\n   📋 ANÁLISIS POR SUCURSAL")
    print(f"   {'─'*40}")
    por_sucursal = conciliacion_final.groupby('sucursal_nombre')['estado'].value_counts().unstack(fill_value=0)
    print(por_sucursal.head(10))

    print(f"\n   📋 TOP 5 SUCURSALES CON MÁS DIFERENCIAS")
    print(f"   {'─'*40}")
    if len(diferencias) > 0:
        top_diff = diferencias.groupby('sucursal_nombre')['diferencia'].agg(['count', 'sum'])
        top_diff.columns = ['num_diferencias', 'monto_total']
        print(top_diff.sort_values('monto_total', ascending=False).head(5).round(2))

    print("\n" + "="*70)
    print("✅ Reporte ejecutivo con datos reales completado")
else:
    print("\n⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
import pandas as pd
import numpy as np

print("📊 CONCILIACIÓN CON TOLERANCIA DE DIFERENCIAS")
print("="*70)

# Libro con errores de redondeo
libro = pd.DataFrame({
    'ID': [1, 2, 3, 4],
    'Concepto': ['Venta A', 'Venta B', 'Venta C', 'Venta D'],
    'Monto_Libro': [100000.50, 85000.00, 42000.75, 190000.00]
})

# Banco con pequeñas diferencias
banco = pd.DataFrame({
    'ID': [1, 2, 3, 5],
    'Desc': ['Dep A', 'Dep B', 'Dep C', 'Dep E'],
    'Monto_Banco': [100000.00, 85000.10, 42001.00, 50000.00]
})

print("\n📄 Datos:")
print("\nLibro:")
print(libro)
print("\nBanco:")
print(banco)

print("\n" + "="*70)
print("\n1️⃣  SIN TOLERANCIA (estricto)")
conc = pd.merge(libro, banco, on='ID', how='outer')
conc['Diff'] = abs(conc['Monto_Libro'].fillna(0) - conc['Monto_Banco'].fillna(0))
conc['Estado_Estricto'] = conc['Diff'].apply(lambda x: '✅ OK' if x == 0 else '❌ Error')
print(conc[['ID', 'Monto_Libro', 'Monto_Banco', 'Diff', 'Estado_Estricto']])
print(f"\nConciliados: {(conc['Estado_Estricto'] == '✅ OK').sum()} de {len(conc)}")

print("\n" + "="*70)
print("\n2️⃣  CON TOLERANCIA ($1)")
TOLERANCIA = 1.0
conc['Estado_Tolerante'] = conc['Diff'].apply(
    lambda x: '✅ OK' if x <= TOLERANCIA else '❌ Error'
)
print(conc[['ID', 'Monto_Libro', 'Monto_Banco', 'Diff', 'Estado_Tolerante']])
print(f"\nConciliados: {(conc['Estado_Tolerante'] == '✅ OK').sum()} de {len(conc)}")
print(f"\n👉 Tolerancia de ${TOLERANCIA} permitió conciliar más registros")

print("\n" + "="*70)
print("✅ Conciliación con tolerancia dominada")

## 🎓 Conclusiones del notebook 05_02

### ✅ Lo que aprendiste

1. **Tolerancia de diferencias:**
   - `abs(monto_libro - monto_banco) <= TOLERANCIA`
   - Permite conciliar registros con diferencias de redondeo
   - Más registros conciliados vs match exacto

2. **Matching multi-criterio:**
   - Coincidir por Referencia + Fecha + Monto
   - `pd.merge(..., how='outer', indicator=True)`
   - Maneja timing differences (fechas levemente diferentes)

3. **Clasificación automática de estados:**
   - ✅ Conciliado: Match exacto o con tolerancia
   - ⚠️ Pendiente: Solo en libro
   - ⚠️ Sobrante: Solo en extracto
   - ❌ Diferencia: Montos diferentes > tolerancia

4. **Reporte ejecutivo:**
   - Métricas automáticas (total conciliado, pendiente, diferencias)
   - Exportar partidas para investigación
   - Dashboard de conciliación

---

### 🎯 Reglas de Oro

👉 **Regla #1: Definir tolerancia según contexto**
```python
# Diferencias absolutas
TOLERANCIA = 1.0  # $1 de tolerancia
abs(diff) <= TOLERANCIA

# Diferencias porcentuales
TOLERANCIA_PCT = 0.01  # 1%
abs(diff) / monto_libro <= TOLERANCIA_PCT
```

👉 **Regla #2: Siempre usar indicator=True**
```python
# MALO
pd.merge(libro, banco, on='ID', how='outer')

# BUENO
pd.merge(libro, banco, on='ID', how='outer', indicator=True)
# _merge column: left_only, right_only, both
```

👉 **Regla #3: Clasificar antes de reportar**
```python
# Clasificar antes de mostrar resultados
conc['Estado'] = conc.apply(clasificar_estado, axis=1)
# Luego generar reporte por estado
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Diferencias de centavos | Tolerancia absoluta ($1) |
| Diferencias porcentuales variables | Tolerancia porcentual (0.5%) |
| Fechas levemente diferentes | Matching por fecha ± 1 día |
| Referencias con formato distinto | Coincidencia fuzzy |
| Movimientos solo en libro | Estado: Pendiente |
| Movimientos solo en extracto | Estado: Sobrante |

---

<div style="background: linear-gradient(90deg, #f59e0b 0%, #fbbf24 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔍 ¡Conciliación avanzada dominada!</h3>
  <p><i>"La tolerancia bien calibrada es la diferencia entre conciliar 80% y 99% de los movimientos."</i></p>
</div>